# 01. Предобработка данных

Загрузка, очистка и кодирование данных опроса Google Forms.

**Целевая переменная:** Anxiety_Level (GAD-7 сумма, 0-21 баллов)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [ ]:
df = pd.read_csv('Anxiety_Level_Test_2.csv')
print(f"Размер датасета: {df.shape}")
df.head(2)

## Переименование колонок

In [ ]:
COLUMN_RENAME = {
    'Отметка времени': 'Timestamp',
    'По Вашему *субъективному мнению*, как Вы оцените свой уровень тревожности от 1 до 10 баллов': 'Subjective_Anxiety',
    'За последние 2 недели, как часто Вы были нервны, тревожны или «на взводе»?': 'GAD1_Nervous',
    'За последние 2 недели, как часто Вы не могли остановить или контролировать беспокойство?': 'GAD2_Uncontrollable',
    'За последние 2 недели, как часто Вы чрезмерно беспокоились по разным поводам?': 'GAD3_Excessive_Worry',
    'За последние 2 недели, как часто Вам было трудно расслабиться?': 'GAD4_Relax',
    'За последние 2 недели, как часто Вы были настолько беспокойны, что Вам было трудно сидеть на месте?': 'GAD5_Restless',
    'За последние 2 недели, как часто Вы были раздражительны?': 'GAD6_Irritable',
    'За последние 2 недели, как часто Вы испытывали страх, будто может случиться что-то плохое?': 'GAD7_Fear',
    'Сколько часов сна Вы в среднем получаете за ночь (последняя неделя)?': 'Sleep_Hours',
    'Физическая активность/Занятие спортом (в среднем часов за неделю)': 'Physical_Activity',
    'Как часто Вы употребляете алкоголь?': 'Alcohol_Frequency',
    'Как Вы опишете свой статус курения?': 'Smoking_Status',
    'Качество вашего питания (самооценка)': 'Diet_Quality',
    'Сколько раз Вы посещали психолога/психотерапевта за последний месяц?': 'Therapy_Sessions',
    'Принимаете ли Вы сейчас лекарства от тревоги/депрессии или другое психотропное лечение?': 'Medication',
    'Как часто Вы выходите на прогулку/на свежий воздух?': 'Outdoor_Walks',
    'Насколько Вы удовлетворены уровнем вашего общения с друзьями, коллегами и знакомыми?': 'Social_Satisfaction',
    'Насколько Вы удовлетворены собственной текущей рабочей/учебной средой (коллеги, учебный коллектив, условия)?': 'Work_Satisfaction',
    'Какой режим Вам больше подходит?': 'Sleep_Pattern',
    'Насколько Вы чувствуете поддержку со стороны близких (друзей/семьи)?': 'Support_Level',
    'Сколько часов в среднем в день вы проводите в телефоне (не считая работы/учёбы)?': 'Screen_Time',
    'Произошло ли в вашей жизни за последний год значимое негативное событие (потеря работы, развод, смерть близкого, серьёзная болезнь и т.п.)?': 'Negative_Event',
    'Ваш пол': 'Gender',
    'Ваш возраст (лет)': 'Age'
}

df = df.rename(columns=COLUMN_RENAME)
print(f"Переименовано колонок: {len(COLUMN_RENAME)}")

## Кодирование ответов GAD-7

Шкала: 0-Ни разу, 1-Несколько дней, 2-Более половины дней, 3-Почти каждый день

In [ ]:
GAD7_MAP = {
    'Ни разу': 0,
    'Несколько дней': 1,
    'Более половины дней': 2,
    'Почти каждый день': 3
}

gad_cols = ['GAD1_Nervous', 'GAD2_Uncontrollable', 'GAD3_Excessive_Worry', 
            'GAD4_Relax', 'GAD5_Restless', 'GAD6_Irritable', 'GAD7_Fear']

for col in gad_cols:
    df[col] = df[col].map(GAD7_MAP)

df['Anxiety_Level'] = df[gad_cols].sum(axis=1)
print(f"GAD-7 диапазон: {df['Anxiety_Level'].min()} - {df['Anxiety_Level'].max()}")

## Кодирование остальных категориальных признаков

In [ ]:
ALCOHOL_MAP = {
    'Никогда': 0,
    'Реже одного раза в месяц': 1,
    '1-3 раза в месяц': 2,
    '1-2 раза в неделю': 3,
    '3-4 раза в неделю': 4,
    'Почти каждый день': 5
}

SMOKING_MAP = {
    'Не курю': 0,
    'Иногда курю': 1,
    'Ежедневно курю': 2
}

THERAPY_MAP = {
    '0': 0,
    '1': 1,
    '2': 2,
    '3 и больше': 3
}

OUTDOOR_MAP = {
    'Никогда': 0,
    'Не более 3 раз в месяц': 1,
    'Раз в неделю': 2,
    'Несколько раз в неделю': 3,
    'Почти каждый день': 4
}

SATISFACTION_MAP = {
    'Совершенно не удовлетворён(а)': 0,
    'Скорее не удовлетворён(а)': 1,
    'Нейтрально': 2,
    'Скорее удовлетворён(а)': 3,
    'Полностью удовлетворён(а)': 4
}

SLEEP_PATTERN_MAP = {
    'Жаворонок — мне удобнее вставать рано и ложиться рано': 0,
    'Сова — мне удобнее ложиться поздно и вставать поздно': 2,
    'Нейтрально — нет яркого предпочтения': 1
}

SUPPORT_MAP = {
    'Никогда': 0,
    'Редко': 1,
    'Иногда': 2,
    'Часто': 3,
    'Всегда': 4
}

EVENT_MAP = {
    'Нет': 0,
    'Да': 1
}

MEDICATION_MAP = {
    'Нет': 0,
    'Да': 1
}

GENDER_MAP = {
    'Мужской': 0,
    'Предпочитаю не говорить': 0.5,
    'Женский': 1
}

df['Alcohol_Frequency'] = df['Alcohol_Frequency'].map(ALCOHOL_MAP)
df['Smoking_Status'] = df['Smoking_Status'].map(SMOKING_MAP)
df['Therapy_Sessions'] = df['Therapy_Sessions'].map(THERAPY_MAP).fillna(0)
df['Outdoor_Walks'] = df['Outdoor_Walks'].map(OUTDOOR_MAP)
df['Social_Satisfaction'] = df['Social_Satisfaction'].map(SATISFACTION_MAP)
df['Work_Satisfaction'] = df['Work_Satisfaction'].map(SATISFACTION_MAP)
df['Sleep_Pattern'] = df['Sleep_Pattern'].map(SLEEP_PATTERN_MAP)
df['Support_Level'] = df['Support_Level'].map(SUPPORT_MAP)
df['Negative_Event'] = df['Negative_Event'].map(EVENT_MAP)
df['Medication'] = df['Medication'].map(MEDICATION_MAP)
df['Gender'] = df['Gender'].map(GENDER_MAP)

print("Категориальные признаки закодированы")

## Итоговый датасет

In [ ]:
FEATURE_COLS = [
    'Age', 'Gender', 'Sleep_Hours', 'Physical_Activity', 'Alcohol_Frequency',
    'Smoking_Status', 'Diet_Quality', 'Therapy_Sessions', 'Medication',
    'Outdoor_Walks', 'Social_Satisfaction', 'Work_Satisfaction', 
    'Sleep_Pattern', 'Support_Level', 'Screen_Time', 'Negative_Event'
]

TARGET_COL = 'Anxiety_Level'

df_clean = df[FEATURE_COLS + [TARGET_COL, 'Subjective_Anxiety']].copy()
df_clean = df_clean.dropna()

print(f"Итоговый размер: {df_clean.shape}")
print(f"\nПроверка типов данных:")
print(df_clean.dtypes)

In [ ]:
df_clean.to_csv('data/processed_data.csv', index=False)
print("Сохранено в: data/processed_data.csv")

In [ ]:
df_clean.describe()